In [1]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/Users/sarahmay/credit_card_fraud_detection_ulb")
OUTPUTS = PROJECT / "outputs"

print("Project exists:", PROJECT.exists())
print("Baseline results:", (OUTPUTS / "phase_4_4/validation_comparison_threshold_0_5.csv").exists())
print("Class-weighted results:", (OUTPUTS / "class_weighted/class_weighted_validation_metrics.csv").exists())
print("SMOTE results:", (OUTPUTS / "smote/smote_validation_metrics.csv").exists())
print("Threshold results:", (OUTPUTS / "phase_4_4/validation_threshold_selection.csv").exists())

Project exists: True
Baseline results: True
Class-weighted results: True
SMOTE results: True
Threshold results: True


In [2]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT = Path("/Users/sarahmay/credit_card_fraud_detection_ulb")
OUTPUTS = PROJECT / "outputs"

# Input files
baseline_path = OUTPUTS / "phase_4_4/validation_comparison_threshold_0_5.csv"
threshold_path = OUTPUTS / "phase_4_4/validation_threshold_selection.csv"
test_path = OUTPUTS / "phase_4_5/final_test_metrics.csv"

# New output folder
RESULTS_DIR = OUTPUTS / "consolidated_research_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Consolidate validation metrics at threshold 0.5
# --------------------------------------------------
validation = pd.read_csv(baseline_path)

validation_columns = [
    "model", "strategy", "average_precision", "roc_auc",
    "threshold", "precision", "recall", "f1",
    "tn", "fp", "fn", "tp", "flagged_count"
]

validation_table = validation[validation_columns].copy()

# Sort for easy comparison
model_order = ["Logistic Regression", "Decision Tree", "Random Forest"]
strategy_order = ["Baseline", "Class-weighted", "SMOTE"]

validation_table["model"] = pd.Categorical(
    validation_table["model"], categories=model_order, ordered=True
)
validation_table["strategy"] = pd.Categorical(
    validation_table["strategy"], categories=strategy_order, ordered=True
)

validation_table = validation_table.sort_values(
    ["strategy", "model"]
).reset_index(drop=True)

validation_table.to_csv(
    RESULTS_DIR / "consolidated_validation_metrics.csv", index=False
)

print("VALIDATION RESULTS — THRESHOLD 0.5")
display(validation_table)

# --------------------------------------------------
# 2. Consolidate threshold-selection results
# --------------------------------------------------
thresholds = pd.read_csv(threshold_path)

threshold_columns = [
    "model", "strategy", "selection_rule", "precision_target",
    "threshold", "validation_precision", "validation_recall",
    "validation_f1", "rule_met", "threshold_fp",
    "threshold_fn", "threshold_tp", "threshold_flagged_count"
]

threshold_table = thresholds[threshold_columns].copy()

threshold_table.to_csv(
    RESULTS_DIR / "threshold_selection_summary.csv", index=False
)

print("\nTHRESHOLD-SELECTION RESULTS")
display(threshold_table)

# --------------------------------------------------
# 3. Basic integrity checks
# --------------------------------------------------
print("\nINTEGRITY CHECKS")
print("Validation rows:", len(validation_table))
print("Expected model-strategy combinations: 9")
print("Actual unique combinations:",
      validation_table[["model", "strategy"]].drop_duplicates().shape[0])
print("All validation metrics present:",
      validation_table[
          ["average_precision", "precision", "recall", "f1"]
      ].notna().all().all())
print("Threshold-selection rows:", len(threshold_table))
print("Final test metrics file exists:", test_path.exists())

print("\nSaved files:")
print(RESULTS_DIR / "consolidated_validation_metrics.csv")
print(RESULTS_DIR / "threshold_selection_summary.csv")

VALIDATION RESULTS — THRESHOLD 0.5


,model,strategy,average_precision,roc_auc,threshold,precision,recall,f1,tn,fp,fn,tp,flagged_count
0,Logistic Regression,Baseline,0.650716,0.968406,0.5,0.658537,0.473684,0.551020,56890,14,30,27,41
1,Decision Tree,Baseline,0.282910,0.824140,0.5,0.435294,0.649123,0.521127,56856,48,20,37,85
2,Random Forest,Baseline,0.772913,0.951765,0.5,0.951220,0.684211,0.795918,56902,2,18,39,41
3,Logistic Regression,Class-weighted,0.772483,0.970913,0.5,0.026494,0.894737,0.051463,55030,1874,6,51,1925
4,Decision Tree,Class-weighted,0.331021,0.806754,0.5,0.538462,0.614035,0.573770,56874,30,22,35,65
5,Random Forest,Class-weighted,0.754697,0.930564,0.5,0.953488,0.719298,0.820000,56902,2,16,41,43
6,Logistic Regression,SMOTE,0.774457,0.964356,0.5,0.030120,0.877193,0.058241,55294,1610,7,50,1660
7,Decision Tree,SMOTE,0.095659,0.822672,0.5,0.146825,0.649123,0.239482,56689,215,20,37,252
8,Random Forest,SMOTE,0.753080,0.970006,0.5,0.955556,0.754386,0.843137,56902,2,14,43,45



THRESHOLD-SELECTION RESULTS


,model,strategy,selection_rule,precision_target,threshold,validation_precision,validation_recall,validation_f1,rule_met,threshold_fp,threshold_fn,threshold_tp,threshold_flagged_count
0,Logistic Regression,Baseline,Maximum validation F1,NaN,0.111299,0.656250,0.736842,0.694215,True,22.0,15.0,42.0,64.0
1,Logistic Regression,Baseline,Maximum recall with precision >= 90%,0.90,0.701976,0.909091,0.350877,0.506329,True,2.0,37.0,20.0,22.0
2,Logistic Regression,Baseline,Maximum recall with precision >= 95%,0.95,0.715949,0.950000,0.333333,0.493506,True,1.0,38.0,19.0,20.0
3,Decision Tree,Baseline,Maximum validation F1,NaN,1.000000,0.435294,0.649123,0.521127,True,48.0,20.0,37.0,85.0
4,Decision Tree,Baseline,Maximum recall with precision >= 90%,0.90,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
5,Decision Tree,Baseline,Maximum recall with precision >= 95%,0.95,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN
6,Random Forest,Baseline,Maximum validation F1,NaN,0.425000,0.933333,0.736842,0.823529,True,3.0,15.0,42.0,45.0
7,Random Forest,Baseline,Maximum recall with precision >= 90%,0.90,0.425000,0.933333,0.736842,0.823529,True,3.0,15.0,42.0,45.0
8,Random Forest,Baseline,Maximum recall with precision >= 95%,0.95,0.470000,0.952381,0.701754,0.808081,True,2.0,17.0,40.0,42.0
9,Logistic Regression,Class-weighted,Maximum validation F1,NaN,0.999977,0.891304,0.719298,0.796117,True,5.0,16.0,41.0,46.0



INTEGRITY CHECKS
Validation rows: 9
Expected model-strategy combinations: 9
Actual unique combinations: 9
All validation metrics present: True
Threshold-selection rows: 27
Final test metrics file exists: True

Saved files:
/Users/sarahmay/credit_card_fraud_detection_ulb/outputs/consolidated_research_results/consolidated_validation_metrics.csv
/Users/sarahmay/credit_card_fraud_detection_ulb/outputs/consolidated_research_results/threshold_selection_summary.csv


In [3]:
import pandas as pd

# Load the consolidated tables
validation = pd.read_csv(
    OUTPUTS / "consolidated_research_results/consolidated_validation_metrics.csv"
)
thresholds = pd.read_csv(
    OUTPUTS / "consolidated_research_results/threshold_selection_summary.csv"
)

# Criterion 1: highest Average Precision
best_ap = validation.loc[validation["average_precision"].idxmax()]

# Criterion 2: highest F1 at threshold 0.5
best_f1 = validation.loc[validation["f1"].idxmax()]

# Criterion 3: highest recall subject to precision >= 95%
eligible = thresholds[
    (thresholds["precision_target"] == 0.95)
    & (thresholds["rule_met"] == True)
].copy()

best_recall_95 = eligible.loc[
    eligible["validation_recall"].idxmax()
]

print("BEST AVERAGE PRECISION")
print(best_ap[["model", "strategy", "average_precision", "precision", "recall", "f1"]])

print("\nBEST F1 AT THRESHOLD 0.5")
print(best_f1[["model", "strategy", "average_precision", "precision", "recall", "f1"]])

print("\nBEST RECALL WITH PRECISION >= 95%")
print(best_recall_95[
    ["model", "strategy", "selection_rule", "threshold",
     "validation_precision", "validation_recall", "validation_f1"]
])

BEST AVERAGE PRECISION
model                Logistic Regression
strategy                           SMOTE
average_precision               0.774457
precision                        0.03012
recall                          0.877193
f1                              0.058241
Name: 6, dtype: object

BEST F1 AT THRESHOLD 0.5
model                Random Forest
strategy                     SMOTE
average_precision          0.75308
precision                 0.955556
recall                    0.754386
f1                        0.843137
Name: 8, dtype: object

BEST RECALL WITH PRECISION >= 95%
model                                          Random Forest
strategy                                               SMOTE
selection_rule          Maximum recall with precision >= 95%
threshold                                              0.535
validation_precision                                0.955556
validation_recall                                   0.754386
validation_f1                                   

In [4]:
from pathlib import Path

PROJECT = Path("/Users/sarahmay/credit_card_fraud_detection_ulb")

keywords = ("cross_validation", "cross-validation", "fold",
            "stability", "repeated", "cv_summary")

matches = [
    str(p.relative_to(PROJECT))
    for p in PROJECT.rglob("*")
    if p.is_file()
    and not any(part.startswith(".") for part in p.parts)
    and any(k in p.name.lower() for k in keywords)
    and not any(part in {".venv", "__pycache__"} for part in p.parts)
]

print("Potential stability-analysis files:")
print("\n".join(matches) if matches else "No matching files found.")

Potential stability-analysis files:
No matching files found.
